# Cohen MWU power — all cell types

Assesses MWU power (TPR) as a function of CRE activity relative to `minP`, for every cell type in the Cohen dataset.

Cohen is an **episomal** MPRA: all CREs are transfected into all cell types (no integrating bottleneck).
Therefore we run **one set of simulations** with all cell types included, rather than separate per-cell-type runs.
Each simulation draws the same synthetic CREs into all cell types with identical activities (`n_cres` confirmed
uniform across cell types in the empirical data).

Steps:
1. Draw `n_cres` synthetic CREs from uniform(min_activity, max_activity), one fixed at `minP` as reference
2. Replicate the same CREs across all cell types in the ground truth
3. Simulate `n_sims` replicates × `n_library_reps` independent libraries using empirical Cohen bounds
4. Run MWU for all CREs vs. reference (one shared hypothesis set, all cell types)
5. Aggregate and plot per-cell-type power curves

In [ ]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import uuid
import math

In [ ]:
data_root = Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

In [ ]:
cluster = SLURMCluster(
    cores=3,
    memory="15G",
    processes=3,
    worker_extra_args=["--nthreads", "2"],
    job_extra_directives=[
        "-p priority",
        "-A prio_skr2",
        "--job-name=simclust_worker",
        "--time=6:00:00",
        "--output=worker_%j.out"
    ]
)
cluster.scale(jobs=10)
client = Client(cluster)

In [ ]:
client.dashboard_link

In [ ]:
# Load training data to confirm n_cres is uniform across cell types (episomal check)
cohen_data = pd.read_parquet(
    data_root / "cohen_ortho" / "retina_single_counting_u6.scmpra",
    columns=["cell_type", "cre_id"]
)
n_cres_by_ct = cohen_data.groupby("cell_type")["cre_id"].nunique()
print("n_cres per cell type (should be uniform):")
print(n_cres_by_ct)
assert n_cres_by_ct.nunique() == 1, "n_cres not uniform across cell types — episomal assumption violated!"

n_cres = int(n_cres_by_ct.iloc[0])
cell_types = list(scm.COHEN_BOUNDS.cells_per_cell_type.index)
print(f"\nn_cres={n_cres}, cell_types={cell_types}")
del cohen_data

In [ ]:
sim_date      = "2026-03-19"
n_library_reps = 100
n_sims         = 5
minP           = scm.COHEN_BOUNDS.reference_activity
min_activity   = scm.COHEN_BOUNDS.min_mpra_umi
max_activity   = minP * 1.05
print(f"minP={minP:.6f}, min={min_activity:.6f}, max={max_activity:.6f}")

In [ ]:
# Cohen is episomal: same CREs transfected into all cell types.
# Each library replicate draws one set of CRE activities and replicates them
# across all cell types in the ground truth. One simulation object covers all
# cell types — no per-cell-type loop needed.

sims = []
for i in range(n_library_reps):
    rng = np.random.default_rng()

    # Draw CRE activities (same values shared across all cell types)
    cre_gt = rng.uniform(min_activity, max_activity, size=n_cres - 1)
    cre_gt = np.append(cre_gt, minP)
    names  = [f"synthcre_{j}" for j in range(n_cres - 1)] + ["reference"]

    # Replicate across all cell types
    gt_df = pd.concat([
        pd.DataFrame({"cre_id": names, "mu": cre_gt, "cell_type": ct})
        for ct in cell_types
    ], ignore_index=True)

    # One library per replicate (unique CRE names only — episomal, shared library)
    libraries = [
        scm.simulate_library(CREs=pd.Series(names), library_model=scm.COHEN_BOUNDS.library_model)
        for _ in range(n_sims)
    ]

    sim = scm.de_novo_simulation(
        location=data_root / f"{sim_date}_cohen_pow",
        name=f"sim_{uuid.uuid4().hex[:8]}",
        client=client,
        libraries=libraries,
        library_mapping="corresponding",
        flatten_overtransfection=True,
        n_sims=n_sims,
        experiment_bounds=scm.COHEN_BOUNDS,
        ground_truth=gt_df,
    )
    sim.gamut()
    sims.append(sim)

print(f"Queued {len(sims)} sims.", flush=True)

In [ ]:
for sim in sims:
    sim.save()
print("Saved.", flush=True)

In [ ]:
# Build one hypothesis set covering all cell types from the first sim
example_data = scm.scMPRA_data.from_parquet(sims[0].scmpradatp / "0.scmpra")
hs_all_ct = scm.make_all_by_celltype_hypotheses(
    counts=example_data,
    reference_cre="reference",
)

for sim in sims:
    sim.add_hypothesis_set("hs_all_ct", hs_all_ct)
    sim.mwu("hs_all_ct")

for sim in sims:
    sim.save()
print("MWU done and saved.", flush=True)

In [ ]:
# Aggregate results, preserving cell type (sum_pow drops it, so we roll our own)
def sum_pow_by_ct(sims, hypothesis_set_name, test_type):
    """Like sum_pow but returns a dict of {cell_type -> DataFrame(reject_null, fc)}."""
    rows = []
    for sim in sims:
        for i in range(sim.get_state_field("n_sims")):
            m = sim._merge_in_ground_truth(hypothesis_set_name, test_type, i)
            m["fc"] = m["comparison_truth"] / m["reference_truth"]
            rows.append(m[["comparison_cell_type", "reject_null", "fc"]])
    combined = pd.concat(rows, ignore_index=True)
    return {
        ct: combined[combined["comparison_cell_type"] == ct][["reject_null", "fc"]].reset_index(drop=True)
        for ct in cell_types
    }

all_mergy = sum_pow_by_ct(sims, "hs_all_ct", "mwu")
print({ct: len(df) for ct, df in all_mergy.items()})

In [ ]:
def _pow_curve_data(mergy, n_bins=100):
    df = mergy.copy()
    df["fc"] = pd.cut(df["fc"], bins=n_bins)
    binned = (
        df.groupby("fc", observed=True)["reject_null"]
        .mean()
        .reset_index(name="reject_frac")
    )
    binned["bin_center"] = binned["fc"].apply(lambda x: x.mid)
    return binned

ncols = 2
nrows = math.ceil(len(cell_types) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), sharey=True)
axes = axes.flatten()
palette = sns.color_palette("tab10", n_colors=len(cell_types))

for i, ct in enumerate(cell_types):
    binned = _pow_curve_data(all_mergy[ct])
    ax = axes[i]
    ax.plot(binned["bin_center"], binned["reject_frac"],
            color=palette[i], marker="o", markersize=2, linewidth=1)
    ax.axhline(0.8, color="black", linestyle="--", lw=0.8)
    ax.axvline(1.0, color="grey",  linestyle=":",  lw=0.8)
    ax.set_title(ct, fontsize=9)
    ax.set_xlabel("FC (activity / minP)", fontsize=8)
    ax.set_ylabel("Power (TPR)", fontsize=8)
    ax.set_ylim(0, 1)
    cells = scm.COHEN_BOUNDS.cells_per_cell_type.get(ct, "?")
    ax.text(0.97, 0.05,
            f"n_cres={n_cres}, cells={cells}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=6, color="grey")

for j in range(len(cell_types), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("MWU Power by Cell Type — Cohen (episomal)", fontsize=12)
plt.tight_layout()
svg_path = output_dir / "power_mwu_all_cell_types.svg"
fig.savefig(svg_path, format="svg", bbox_inches="tight")
print(f"Saved: {svg_path}")
plt.show()

In [ ]:
client.close()
cluster.close()